In [ ]:
def printmd(string):
    display(Markdown(string))

def display_results(results_df):
  for task_name, df in results_df.items():
    printmd(f"# {task_name}")
    printmd("---")
    display(HTML(df.to_html()))

In [ ]:
# PREPROCESS_FOLD---------------------------------------------------------------------------------------------------------------
def check_node_type_distribution(X_train, X_val, X_test, node_col='node_type', normalize=True): # used in preprocessing(not used in kan) and preprocess_fold
    """
    Check and compare the distribution of `node_type` (or any categorical feature)
    across train, validation, and test splits.

    Parameters
    ----------
    X_train, X_val, X_test : pd.DataFrame
        DataFrames for training, validation, and test sets.
    node_col : str, default='node_type'
        Name of the column representing node types (or any categorical variable).
    normalize : bool, default=True
        If True, show proportions (%) instead of raw counts.

    Returns
    -------
    pd.DataFrame
        Summary table comparing distributions across splits.
    """
    # Validate column existence
    for split_name, df in zip(['Train', 'Val', 'Test'], [X_train, X_val, X_test]):
        if node_col not in df.columns:
            raise ValueError(f"'{node_col}' not found in {split_name} set.")

    # Compute distributions
    def get_dist(df, name):
        return df[node_col].value_counts(normalize=normalize).rename(name)

    dist_train = get_dist(X_train, 'Train')
    dist_val   = get_dist(X_val, 'Val')
    dist_test  = get_dist(X_test, 'Test')

    # Combine into one comparison DataFrame
    dist_summary = pd.concat([dist_train, dist_val, dist_test], axis=1).fillna(0)
    if normalize:
        dist_summary = dist_summary * 100  # show as %

    # Add total rows for reference
    total_counts = {
        "Train Total": len(X_train),
        "Val Total": len(X_val),
        "Test Total": len(X_test)
    }
    totals_df = pd.DataFrame(total_counts, index=["Total Rows"])

    printmd("## Node Type Distribution Across Splits:")
    print(dist_summary.round(2))
    print("\n Dataset Sizes:")
    print(totals_df)

    return dist_summary



def check_data_leakage(X_train, X_val, X_test, group_cols=None): # used in preprocessing and preprocess_fold
    """
    Verify that there is no data leakage (i.e., no repeated feature combinations)
    across train, validation, and test splits.

    Parameters
    ----------
    X_train, X_val, X_test : pd.DataFrame
        DataFrames for train, validation, and test sets.
    group_cols : list of str, optional
        Columns to consider for checking duplicates.
        If None, all columns of X_train are used.

    Returns
    -------
    None
        Prints leakage report. Raises AssertionError if leakage is detected.
    """


    if group_cols is None:
        group_cols = X_train.columns.tolist()

    # Create a unique signature for each feature combination
    def make_signatures(df):
        return set(df[group_cols].astype(str).agg('-'.join, axis=1))

    sig_train = make_signatures(X_train)
    sig_val   = make_signatures(X_val)
    sig_test  = make_signatures(X_test)

    # Check overlaps
    leak_train_val = sig_train.intersection(sig_val)
    leak_train_test = sig_train.intersection(sig_test)
    leak_val_test = sig_val.intersection(sig_test)

    total_leaks = len(leak_train_val) + len(leak_train_test) + len(leak_val_test)

    printmd("## Data Leakage Check:")

    print(f"Train Samples: {len(X_train)}")
    print(f"Val Samples: {len(X_val)}")
    print(f"Test Samples: {len(X_test)}")

    print("\n")
    # print(f"- Train–Val overlaps: {len(leak_train_val)}")
    # print(f"- Train–Test overlaps: {len(leak_train_test)}")
    # print(f"- Val–Test overlaps: {len(leak_val_test)}")

    if total_leaks == 0:
        print("No data leakage detected. Splits are clean.")
    else:
        print("Data leakage detected!")
        if len(leak_train_val):
            print(f"  → {len(leak_train_val)} overlapping feature groups between Train and Val")
        if len(leak_train_test):
            print(f"  → {len(leak_train_test)} overlapping feature groups between Train and Test")
        if len(leak_val_test):
            print(f"  → {len(leak_val_test)} overlapping feature groups between Val and Test")
        #raise AssertionError("Data leakage detected between splits.")

def fit_minmax_scaler(X, y=None, regression_cols=None, classification_col=None): # used in preprocessing(not used in kan notebook) and preprocess_fold and prepare_cross_domain_evaluation_data(not used in the notebook)
    """
    Fit MinMaxScaler for features (and optionally regression targets).

    X:
      - node_type column is excluded from scaling

    y:
      - only regression columns are scaled
      - classification column remains unchanged
    """

    # ----------------------
    # 1. Handle feature scaler
    # ----------------------
    scaler_x = MinMaxScaler(feature_range=(1, 2))
    # Drop node_type if exists
    if "node_type" in X.columns:
        scaler_x.fit(X.drop(columns=["node_type"]))
    else:
        scaler_x.fit(X)

    # ----------------------
    # 2. Handle target scaler
    # ----------------------
    scaler_y = None
    if y is not None and regression_cols is not None:
        # Fit scaler on regression columns only
        scaler_y = MinMaxScaler(feature_range=(1, 2))
        scaler_y.fit(y[regression_cols])

    # ---- Logging ----
    print("[INFO] MinMaxScaler fitted:")
    print(f"  - Features shape: {X.shape}")
    if y is not None:
        print(f"  - Regression targets shape: {y[regression_cols].shape}")
        print(f"  - Classification column '{classification_col}' excluded from scaling")
    print()

    return scaler_x, scaler_y


def save_scalers(scaler_x, scaler_y, task_name, base_path='./scalers/functions/'): # used in preprocessing(not used in kan notebook) and preprocess_fold and prepare_cross_domain_evaluation_data(not used in the notebook)
    """
    Save fitted scalers for features and targets.

    Parameters
    ----------
    scaler_x : MinMaxScaler
        Fitted feature scaler.
    scaler_y : MinMaxScaler or None
        Fitted target scaler (if any).
    task_name : str
        Name of the task (e.g., 'Multi_Target_regression', 'overloaded_node_classification')
    base_path : str, default='./scalers/functions/'
        Base directory to save scalers.
    """
    # Create directories
    x_dir = os.path.join(base_path, 'scaler_x')
    y_dir = os.path.join(base_path, 'scaler_y')
    os.makedirs(x_dir, exist_ok=True)
    os.makedirs(y_dir, exist_ok=True)

    # Save X scaler
    joblib.dump(scaler_x, os.path.join(x_dir, f"{task_name}_x.joblib"))

    # Save Y scaler (only if provided)
    if scaler_y is not None:
        joblib.dump(scaler_y, os.path.join(y_dir, f"{task_name}_y.joblib"))

    print(f"[INFO] Scalers saved for target '{task_name}'")
    if scaler_y is None:
        print("  - No target scaler (classification or categorical target).")
    print()

def transform_with_scalers( # used in preprocessing and preprocess_fold 
    X_train, X_val, X_test, scaler_x,
    y_train=None, y_val=None, y_test=None,
    scaler_y=None, regression_cols=None, classification_col=None
):
    """
    Transform train/val/test using fitted scalers.

    X:
      - scales all columns except node_type
      - adds node_type back after scaling

    y:
      - scales only regression targets
      - keeps classification target unchanged
    """

    # =========================================
    # 1. Separate node_type from X
    # =========================================
    def split_node_type(df):
        if "node_type" in df.columns:
            node = df["node_type"].values.reshape(-1, 1)
            df_wo = df.drop(columns=["node_type"])
        else:
            node = None
            df_wo = df
        return df_wo, node


    X_train_noNT, node_train = split_node_type(X_train)
    X_val_noNT, node_val = split_node_type(X_val)
    X_test_noNT, node_test = split_node_type(X_test)

    # =========================================
    # 2. Scale numeric features only
    # =========================================
    X_train_scaled = scaler_x.transform(X_train_noNT)
    X_val_scaled   = scaler_x.transform(X_val_noNT)
    X_test_scaled  = scaler_x.transform(X_test_noNT)

    # Add back node_type (not scaled)
    if node_train is not None:
        X_train_scaled = np.hstack([node_train, X_train_scaled])
        X_val_scaled   = np.hstack([node_val, X_val_scaled ])
        X_test_scaled  = np.hstack([node_test, X_test_scaled])

    # =========================================
    # 3. Handle Y scaling
    # =========================================
    if y_train is not None and scaler_y is not None:

        # Extract and scale regression columns
        y_train_reg_s = scaler_y.transform(y_train[regression_cols])
        y_val_reg_s   = scaler_y.transform(y_val[regression_cols])
        y_test_reg_s  = scaler_y.transform(y_test[regression_cols])

        # Recombine regression + classification
        y_train_scaled = pd.DataFrame(y_train_reg_s, columns=regression_cols)
        y_val_scaled   = pd.DataFrame(y_val_reg_s, columns=regression_cols)
        y_test_scaled  = pd.DataFrame(y_test_reg_s, columns=regression_cols)

        # Add classification target unchanged
        y_train_scaled[classification_col] = y_train[classification_col].values
        y_val_scaled[classification_col]   = y_val[classification_col].values
        y_test_scaled[classification_col]  = y_test[classification_col].values

    else:
        y_train_scaled, y_val_scaled, y_test_scaled = y_train, y_val, y_test

    # ---- Logging ----
    print("[INFO] Transformation complete.")
    print(f"  - X_train shape: {X_train_scaled.shape}")
    print(f"  - X_val shape: {X_val_scaled.shape}")
    print(f"  - X_test shape: {X_test_scaled.shape}\n")

    return (
        X_train_scaled, X_val_scaled, X_test_scaled,
        y_train_scaled, y_val_scaled, y_test_scaled
    )

def transform(df, rate_columns=None, pad_value=0.0): # used in preprocessing and preprcess_fold and prepare_cross_domain_evaluation_data(not used in the notebook)
    """
    Transform the input dataframe into padded, variable-length sequences.

    Each sample becomes a sequence of steps where each step = [rate, method_onehot(6), node_type].
    Steps with zero rate are removed. Then sequences are padded to the same length.

    Returns:
        X_padded: np.ndarray, shape (num_samples, max_seq_len, 8)
        mask: np.ndarray, shape (num_samples, max_seq_len), 1 for valid, 0 for padded
    """

    if rate_columns is None:
        rate_columns = [
            'rate_function_env', 'rate_function_curl', 'rate_function_eat_memory',
            'rate_function_nmap', 'rate_function_shasum', 'rate_function_figlet'
        ]

    node_type = df['node_type'].values
    num_samples = df.shape[0]
    sequence_length = len(rate_columns)  # 6

    # One-hot encode method indices (0–5)
    method_ids = np.arange(sequence_length).reshape(-1, 1)
    encoder = OneHotEncoder(sparse_output=False, categories='auto')
    method_onehot = encoder.fit_transform(method_ids)  # (6, 6)

    X_sequences = []

    for i in range(num_samples):
        node_val = node_type[i]
        rate_values = df.loc[i, rate_columns].values  # shape (6,)

        # Keep only non-zero steps
        nonzero_mask = rate_values != 1
        if not np.any(nonzero_mask):
            nonzero_mask = np.array([True])  # Keep one dummy step if all zero

        rate_seq = rate_values[nonzero_mask].reshape(-1, 1)
        method_seq = method_onehot[nonzero_mask]
        node_seq = np.full((np.sum(nonzero_mask), 1), node_val)

        seq = np.concatenate([rate_seq, method_seq, node_seq], axis=1)  # (L_i, 8)
        X_sequences.append(seq)

    # Pad all sequences to same length (post-padding with zeros)
    max_len = max(len(seq) for seq in X_sequences)
    X_padded = pad_sequences(X_sequences, maxlen=max_len, dtype='float32', padding='post', value=pad_value)

    # Create mask (1 for real steps, 0 for padded)
    mask = np.array([[1]*len(seq) + [0]*(max_len - len(seq)) for seq in X_sequences], dtype='float32')

    # Drop node_type column if you want to reuse df
    df = df.drop(columns=['node_type'])

    return X_padded

def preprocess_fold(X_train, X_val, X_test, y_train, y_val, y_test,tasks,base_path): # used in cross_validate
    """
    Perform full preprocessing pipeline:
    - Split data into train/validation/test sets
    - Fit MinMax scalers for features and (optionally) targets
    - Save scalers to disk
    - Transform datasets using fitted scalers
    - Return scaled datasets and scalers for all tasks
    """
    printmd("## Preprocessing")
    # -----------------------------------------------------
    # Initialize dictionaries to store splits and scalers
    # -----------------------------------------------------
    x_train_dict, x_val_dict, x_test_dict = {}, {}, {}
    y_train_dict, y_val_dict, y_test_dict = {}, {}, {}
    x_scalers, y_scalers = {}, {}

    # -----------------------------------------------------
    # Loop through each task for preprocessing
    # -----------------------------------------------------
    for task_name,task_info in tasks.items():
        printmd(f"# {task_name}")
        printmd("---")


        # -------------------------------------------------
        # Check for data leakage
        # -------------------------------------------------
        check_node_type_distribution(X_train,X_val,X_test)
        check_data_leakage(X_train,X_val,X_test)

        # -------------------------------------------------
        # Fit MinMaxScaler
        # Skip target scaling for classification tasks
        # -------------------------------------------------
        printmd("## Scalaing")
        if task_name == "overloaded_node_classification":
            scaler_x, scaler_y = fit_minmax_scaler(X_train)
        else:
            scaler_x, scaler_y = fit_minmax_scaler(X_train, y_train,
                                                   regression_cols=task_info["regression_targets"],
                                                  classification_col=task_info["classification_targets"])

        # -------------------------------------------------
        # Save fitted scalers to disk
        # -------------------------------------------------
        save_scalers(scaler_x, scaler_y, task_name, base_path)

        # -------------------------------------------------
        # Apply scaling to train/val/test sets
        # -------------------------------------------------
        (
            X_train_scaled, X_val_scaled, X_test_scaled,
            y_train_scaled, y_val_scaled, y_test_scaled
        ) = transform_with_scalers(
            X_train, X_val, X_test,
            scaler_x,
            y_train, y_val, y_test,
            scaler_y,
            regression_cols=task_info["regression_targets"],
            classification_col=task_info["classification_targets"]
        )


        # -------------------------------------------------
        # Optional: Apply any custom transformation on scaled features
        # Skip this step for classification tasks
        # -------------------------------------------------
        print(f"Transforming for Janoosy: {datetime.datetime.now()}")
        if task_name != "overloaded_node_classification":
            X_train_scaled = transform(pd.DataFrame(X_train_scaled, columns=features))
            X_val_scaled = transform(pd.DataFrame(X_val_scaled, columns=features))
            X_test_scaled = transform(pd.DataFrame(X_test_scaled, columns=features))

        # -------------------------------------------------
        # Store scaled datasets and scalers in dictionaries
        # -------------------------------------------------
        x_train_dict[task_name] = X_train_scaled
        x_val_dict[task_name] = X_val_scaled
        x_test_dict[task_name] = X_test_scaled
        x_scalers[task_name] = scaler_x

        y_train_dict[task_name] = y_train_scaled
        y_val_dict[task_name] = y_val_scaled
        y_test_dict[task_name] = y_test_scaled
        y_scalers[task_name] = scaler_y

    # -----------------------------------------------------
    # Return all processed splits and scalers
    # -----------------------------------------------------
    return (
        x_train_dict, x_val_dict, x_test_dict, x_scalers,
        y_train_dict, y_val_dict, y_test_dict, y_scalers
    )


In [ ]:
# perform_custom_oversampling (used in cross_validate) ----------------------------------------------------------------------------------------------------------
def limit_identical_combinations(df, features, target_cols, max_per_combination=1): # used in perform_custom_oversampling
    """
    Limit the dataset to at most `max_per_combination` identical feature combinations.

    Parameters
    ----------
    df : pd.DataFrame
    features : list
        Columns to group by when identifying identical combinations.
    target_cols : list
        Target column names.
    max_per_combination : int
        Maximum allowed identical combinations per group.

    Returns
    -------
    df_limited : pd.DataFrame
        Reduced datasets after limiting duplicates.
    """
    print(f"\n=== Limiting to max {max_per_combination} identical combinations ===")

    # df_combined = pd.concat([X, y], axis=1)
    df_limited = (
        df.groupby(features, as_index=False)
        .head(max_per_combination)
        .reset_index(drop=True)
    )
# old version
    #df_limited = (
        #df.groupby(features, group_keys=False)
        #.apply(lambda x: x.head(max_per_combination))
        #.reset_index(drop=True)
    #)

    X_limited = df_limited[features]
    y_limited = df_limited[target_cols]

    print(f"Original row count: {len(df)}")
    print(f"After limiting: {len(df_limited)} (removed {len(df) - len(df_limited)})")
    print("\nClass distribution after limiting:")
    print(y_limited.value_counts())

    return df_limited


def generate_synthetic_overloaded(df, features, target_col): # used in perform_custom_oversampling
    """
    Generate synthetic overloaded samples to balance classes per node_type.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataset (after limiting).
    features : list
        Feature column names.
    target_col : str
        Binary target column name (0 = non-overloaded, 1 = overloaded).

    Returns
    -------
    df_augmented : pd.DataFrame
        Dataset containing newly generated synthetic rows.
    """
    print("\n=== Generating synthetic overloaded samples (Unique Features Balancing) ===")

    rate_cols = [col for col in df.columns if col.startswith("rate_function_")]
    node_types = df["node_type"].unique()
    augmented_rows = []
    generated_total = 0

    existing_keys = set(tuple(row[col] for col in features) for _, row in df.iterrows())

    for nt in node_types:
        subset = df[df["node_type"] == nt]
        count_1 = subset[subset[target_col] == 1].shape[0]
        count_0 = subset[subset[target_col] == 0].shape[0]
        needed = count_0 - count_1

        print(f"\n[Node Type {nt}] Overloaded: {count_1}, Non-overloaded: {count_0}")
        if needed <= 0:
            print(f"Already balanced or overloaded.")
            continue

        print(f"Generating {needed} synthetic samples for node_type {nt}")
        df_overloaded = subset[subset[target_col] == 1]
        generated = 0

        for _, row in df_overloaded.iterrows():
            overloaded_funcs = [col for col in rate_cols if row[col] > 0]
            if len(overloaded_funcs) != 3:
                continue

            fixed_rates = {col: row[col] for col in overloaded_funcs}

            for target_func in overloaded_funcs:
                start = int(fixed_rates[target_func])
                for val in range(start + 5, 201, 10):
                    if generated >= needed:
                        break

                    new_row = {col: 0.0 for col in rate_cols}
                    for col in overloaded_funcs:
                        new_row[col] = val if col == target_func else fixed_rates[col]

                    # Fill metadata and targets
                    new_row["node_type"] = nt
                    new_row[target_col] = 1

                    # Add other columns from original
                    for col in df.columns:
                        if col not in new_row and col not in rate_cols:
                            new_row[col] = row[col]

                    new_key = tuple(new_row[col] for col in features)
                    if new_key not in existing_keys:
                        augmented_rows.append(new_row)
                        existing_keys.add(new_key)
                        generated += 1
                        generated_total += 1

                if generated >= needed:
                    break
            if generated >= needed:
                break

        print(f"Generated {generated} samples for node_type {nt}")

    print(f"\n Total synthetic samples generated: {generated_total}")
    return pd.DataFrame(augmented_rows)


def balance_df_nodewise(df, df_augmented, node_col='node_type', target_col='overloaded_node'): # used in perform_custom_oversampling
    """
    Balance df within each node_type based on overloaded_node (target).
    df_augmented contains only rows with overloaded_node = 1.
    Includes detailed debug prints.
    """

    balanced_parts = []
    node_types = df[node_col].unique()

    print("\n========= ROWS BALANCING =========\n")
    print(f"Node types found: {list(node_types)}\n")

    for nt in node_types:
        print(f"\n--- Processing node_type: {nt} ---")

        subset = df[df[node_col] == nt].copy()

        # Count original samples
        count_1 = (subset[target_col] == 1).sum()
        count_0 = (subset[target_col] == 0).sum()

        print(f"Original counts → overloaded=0: {count_0}, overloaded=1: {count_1}")

        # Check if balance is needed
        if count_1 >= count_0:
            print("Already balanced or overloaded=1 is majority. No augmentation needed.")
            balanced_parts.append(subset)
            continue

        needed = count_0 - count_1
        print(f"Need to add {needed} more rows for overloaded=1")

        # Filter augmented rows for this node_type
        aug_rows = df_augmented[df_augmented[node_col] == nt]

        if aug_rows.empty:
            print(f"[WARNING] df_augmented has NO rows for node_type={nt}. Cannot balance this group!")
            balanced_parts.append(subset)
            continue

        print(f"Augmented rows available for this node_type={nt}: {len(aug_rows)}")

        # Calculate repeat factor
        repeats = ceil(needed / len(aug_rows))
        print(f"Repeating augmented rows {repeats} times")

        repeated = pd.concat([aug_rows] * repeats, ignore_index=True)

        rows_to_add = repeated.iloc[:needed]
        print(f"Rows actually added: {len(rows_to_add)}")

        # Append augmented rows
        subset = pd.concat([subset, rows_to_add], ignore_index=True)

        # Verify new counts
        new_count_1 = (subset[target_col] == 1).sum()
        new_count_0 = (subset[target_col] == 0).sum()
        print(f"After augmentation → overloaded=0: {new_count_0}, overloaded=1: {new_count_1}")

        balanced_parts.append(subset)

    print("\n========= BALANCING COMPLETED =========\n")
    final_df = pd.concat(balanced_parts, ignore_index=True)
    return final_df


def perform_custom_oversampling(df, features, reg_target, class_target): # used in prepare_feature_target_datasets(not in the notebook) and cross_validate
    """
    Custom domain-aware oversampling replacing SMOTENC.

    Steps:
    1. Limit identical feature combinations.
    2. Generate synthetic overloaded samples per node_type to balance unique features(Group level balance).
    3. Row-level Balance by duplicating augmented rows and returning a balanced dataset.

    Parameters
    ----------
    df : pd.DataFrame
    features : list
        Feature columns used for grouping and synthesis.
    reg_target : list
        Regression Target column(s).
    class_target : list
        Classification Target column(s).

    Returns
    -------
    X_resampled, y_resampled : pd.DataFrame, pd.DataFrame
        Balanced feature and target datasets.
    """
    printmd("## Oversampling")
    # Step 1: Apply limiting rule
    df_class = df[features + class_target]
    df_limited = limit_identical_combinations(df_class, features, class_target)

    # Step 2: Combine for rebalancing
    # df_limited = pd.concat([X_limited, y_limited], axis=1)
    target_col = class_target[0]

    # Step 3: Generate new samples to balance limited dataset
    df_augmented = generate_synthetic_overloaded(df_limited, features, target_col)

    # Optionally save the augmented rows
    os.makedirs(PATH_TO_AUGMENTED_ROWS, exist_ok=True)
    df_augmented.to_csv(PATH_TO_AUGMENTED_ROWS + "augmented_overloaded_samples.csv", index=False)
    print("Augmented rows saved to 'augmented_overloaded_samples.csv'")

    # Step 4: Final overall balance dataset (repeated features)
    df_balanced = balance_df_nodewise(df, df_augmented)
    X_res = df_balanced[features]
    y_res = df_balanced[reg_target + class_target]

    print("\nFinal class distribution after rebalancing:")
    print(y_res[class_target].value_counts())
    print("\nFinal class distribution after rebalancing (node_type-wise):")
    print(df_balanced.groupby("node_type")["overloaded_node"].value_counts().unstack(fill_value=0))

    print("\nFinal node_type distribution:")
    print(X_res["node_type"].value_counts())

    print("\n========= FINAL UNIQUE FEATURES SUMMARY =========")

    # Convert to tuples for unique combinations
    all_unique = df_balanced[features].drop_duplicates()
    unique_0 = df_balanced[df_balanced[target_col] == 0][features].drop_duplicates()
    unique_1 = df_balanced[df_balanced[target_col] == 1][features].drop_duplicates()

    print(f"Total unique feature combinations       : {len(all_unique)}")
    print(f"Unique feature combinations for 0       : {len(unique_0)}")
    print(f"Unique feature combinations for 1       : {len(unique_1)}")

    print("==========================================\n")

    return X_res, y_res, df_augmented, df_balanced



In [ ]:
def cross_validate(features_dict, targets_dict, tasks,tasks_unified, cv_splits_path, scaler_path,
                   models_path,logs_path,results_path,plot_path,n_splits=5):

  for task_name, task_info in tasks_unified.items():
    printmd(f"# Task: {task_name}")
    printmd("---")

    X = features_dict[task_name]
    y = targets_dict[task_name]

    # Get folds
    print(f"Start CV Split: {datetime.datetime.now()}")
    splits = get_folds(X, y, n_splits)

    all_results = []
    training_times = {}
    # nodewise_results = []

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        printmd(f"# ===== FOLD {fold} / {n_splits} ====")
        printmd("---")

        # Slice for outer train and test
        X_train_outer, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train_outer, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # =====================================
        # Inner split: train → train/val
        # =====================================
        print(f"Start Train Val Split {fold}: {datetime.datetime.now()}")
        inner_train_idx, inner_val_idx = stratified_group_shuffle_split(
            X_train_outer, # Pass the outer train set
            y_train_outer.values[:, -1].astype(int)
        )

        X_train = X_train_outer.loc[inner_train_idx]
        y_train = y_train_outer.loc[inner_train_idx]
        X_val = X_train_outer.loc[inner_val_idx]
        y_val = y_train_outer.loc[inner_val_idx]

        print(f"Start Oversampling {fold}: {datetime.datetime.now()}")
        # Oversample train only
        df = pd.concat([X_train, y_train], axis=1)
        X_train, y_train, df_augmented, df_balanced = perform_custom_oversampling(df, task_info["features"], task_info["regression_targets"],
                                                         task_info["classification_targets"])

        save_xy_split(X_train, y_train, X_val, y_val, X_test, y_test, cv_splits_path+f"fold_{fold}")

        print(f"Start Preprocessing {fold}: {datetime.datetime.now()}")
        #  Preprocess
        (
        x_train_dict, x_val_dict, x_test_dict, x_scalers,
        y_train_dict, y_val_dict, y_test_dict, y_scalers
    )      = preprocess_fold(X_train, X_val, X_test, y_train, y_val, y_test,tasks_unified,scaler_path+f"fold_{fold}")

        print(f"Start Training {fold}: {datetime.datetime.now()}")
        #  Train
        trained_models, models_history, training_time = training(x_train_dict, x_val_dict, y_train_dict,y_val_dict, tasks_unified,
                                                        base_path= models_path + f"fold_{fold}/",
                                                        logs_path = logs_path + f"fold_{fold}/")
        print(f"End Training {fold}: {datetime.datetime.now()}")

        plot_training_histories_separate(models_history,metric_map=metric_map_loss,save_dir = plot_path + f"/fold_{fold}")
        plot_training_histories_separate(models_history,metric_map=metric_map_r2, save_dir = plot_path + f"/fold_{fold}")

        #  Evaluate
        all_predictions = predict(trained_models, x_test_dict, tasks_unified)

        printmd(f"# Fold {fold} Results:")
        results_df_source = show_result(all_predictions,y_test_dict,tasks,
                                base_path= results_path + f"fold_{fold}/",
                                plot_path=plot_path + f"/fold_{fold}")

        all_results.append(results_df_source)
        training_times[fold] = training_time


        # Evaluate Nodewise
        # x_test_nodewise , y_test_nodewise =  split_test_nodewise(x_test_dict,y_test_dict,tasks_unified)
        # results_df_nodewise = nodewise_evaluation(trained_models, x_test_nodewise, y_test_nodewise, tasks,
        #                                          tasks_unified,
        #             PATH_TO_RESULTS +f"source/all_node/nodewise/fold{fold}/")

        # nodewise_results.append(results_df_nodewise)

        # Objects to clean up
        to_delete = [
            trained_models, models_history,
            x_train_dict, x_val_dict, x_test_dict,
            y_train_dict, y_val_dict, y_test_dict,
            X_train_outer, X_test, X_train, X_val,
            y_train_outer, y_test, y_train, y_val,
            df, df_augmented, df_balanced
        ]

        # Cleanup memory
        cleanup_fold_memory(to_delete)

    return all_results, training_times

# test new function

In [ ]:
def prepare_single_split_before_training(
    features_dict,
    targets_dict,
    tasks,
    tasks_unified,
    cv_splits_path,
    scaler_path,
    test_size=0.2
):
    all_outputs = {}

    for task_name, task_info in tasks_unified.items():
        printmd(f"# Task: {task_name}")
        printmd("---")

        X = features_dict[task_name]
        y = targets_dict[task_name]

        # =====================================
        # 1. Single split: train_outer / test
        # =====================================
        print(f"Start Train/Test Split: {datetime.datetime.now()}")

        indices = np.arange(len(X))
        train_idx, test_idx = m(
            indices,
            test_size=test_size,
            stratify=y.values[:, -1].astype(int),
            random_state=42
        )

        X_train_outer, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train_outer, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # =====================================
        # 2. Inner split: train / val
        # =====================================
        print(f"Start Train/Val Split: {datetime.datetime.now()}")

        inner_train_idx, inner_val_idx = stratified_group_shuffle_split(
            X_train_outer,
            y_train_outer.values[:, -1].astype(int)
        )

        X_train = X_train_outer.loc[inner_train_idx]
        y_train = y_train_outer.loc[inner_train_idx]
        X_val = X_train_outer.loc[inner_val_idx]
        y_val = y_train_outer.loc[inner_val_idx]

        # =====================================
        # 3. Oversampling only on train
        # =====================================
        print(f"Start Oversampling: {datetime.datetime.now()}")

        df = pd.concat([X_train, y_train], axis=1)
        X_train, y_train, df_augmented, df_balanced = perform_custom_oversampling(
            df,
            task_info["features"],
            task_info["regression_targets"],
            task_info["classification_targets"]
        )

        # =====================================
        # 4. Save raw splits
        # =====================================
        save_xy_split(
            X_train, y_train,
            X_val, y_val,
            X_test, y_test,
            cv_splits_path
        )

        # =====================================
        # 5. Preprocessing
        # =====================================
        print(f"Start Preprocessing: {datetime.datetime.now()}")

        (
            x_train_dict, x_val_dict, x_test_dict, x_scalers,
            y_train_dict, y_val_dict, y_test_dict, y_scalers
        ) = preprocess_fold(
            X_train, X_val, X_test,
            y_train, y_val, y_test,
            tasks_unified,
            scaler_path
        )

        # =====================================
        # 6. Save everything before training
        # =====================================
        all_outputs[task_name] = {
            "X_train_raw": X_train,
            "X_val_raw": X_val,
            "X_test_raw": X_test,
            "y_train_raw": y_train,
            "y_val_raw": y_val,
            "y_test_raw": y_test,
            "df_augmented": df_augmented,
            "df_balanced": df_balanced,
            "x_train_dict": x_train_dict,
            "x_val_dict": x_val_dict,
            "x_test_dict": x_test_dict,
            "y_train_dict": y_train_dict,
            "y_val_dict": y_val_dict,
            "y_test_dict": y_test_dict,
            "x_scalers": x_scalers,
            "y_scalers": y_scalers
        }

    return all_outputs

# test 2

In [ ]:
VERSION = 'V1'
PATH_TO_SCALERS = drive_path + f'output/scalers/baselines/{VERSION}/'
PATH_TO_MODELS = drive_path + f"output/system-forecaster-models/baselines/{VERSION}/"
PATH_TO_RESULTS = drive_path + f"output/results/baselines/{VERSION}/"
PATH_TO_TRAINING_LOGS = drive_path + f"output/training_logs/baselines/{VERSION}/"
PATH_TO_AUGMENTED_ROWS = drive_path + f"data/augmented_rows/baselines/{VERSION}/"
PATH_TO_SPLITS = drive_path + f"data/splits/{VERSION}/"
PATH_TO_PLOTS  = drive_path + f"output/plots/baselines/{VERSION}/"

In [ ]:
def single_split_preprocessing(
    features_dict,
    targets_dict,
    tasks_unified,
    output_path,
    scaler_path,
    n_splits=5,
    random_state=42
):
    all_outputs = {}

    for task_name, task_info in tasks_unified.items():
        printmd(f"# Task: {task_name}")
        printmd("---")

        X = features_dict[task_name]
        y = targets_dict[task_name]

        # =====================================
        # STESSO SPLIT DELLA CROSS_VALIDATE
        # ma uso solo il primo fold
        # =====================================
        print(f"Start CV Split (single fold only): {datetime.datetime.now()}")
        splits = get_folds(X, y, n_splits=n_splits, random_state=random_state)

        train_idx, test_idx = next(splits)

        # Outer split: train/test
        X_train_outer, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train_outer, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # =====================================
        # STESSO SPLIT TRAIN/VAL DELLA CROSS_VALIDATE
        # =====================================
        print(f"Start Train/Val Split: {datetime.datetime.now()}")
        inner_train_idx, inner_val_idx = stratified_group_shuffle_split(
            X_train_outer,
            y_train_outer.values[:, -1].astype(int)
        )

        X_train = X_train_outer.loc[inner_train_idx]
        y_train = y_train_outer.loc[inner_train_idx]
        X_val = X_train_outer.loc[inner_val_idx]
        y_val = y_train_outer.loc[inner_val_idx]

        # =====================================
        # OVERSAMPLING SOLO TRAIN
        # =====================================
        print(f"Start Oversampling: {datetime.datetime.now()}")
        df_train = pd.concat([X_train, y_train], axis=1)

        X_train, y_train, df_augmented, df_balanced = perform_custom_oversampling(
            df_train,
            task_info["features"],
            task_info["regression_targets"],
            task_info["classification_targets"]
        )

        # =====================================
        # SALVA SPLIT RAW
        # =====================================
        save_xy_split(
            X_train, y_train,
            X_val, y_val,
            X_test, y_test,
            output_path
        )

        # =====================================
        # PREPROCESSING
        # =====================================
        print(f"Start Preprocessing: {datetime.datetime.now()}")

        (
            x_train_dict, x_val_dict, x_test_dict, x_scalers,
            y_train_dict, y_val_dict, y_test_dict, y_scalers
        ) = preprocess_fold(
            X_train, X_val, X_test,
            y_train, y_val, y_test,
            tasks_unified,
            scaler_path
        )

        # =====================================
        # OUTPUT
        # =====================================
        all_outputs[task_name] = {
            "X_train_raw": X_train,
            "X_val_raw": X_val,
            "X_test_raw": X_test,
            "y_train_raw": y_train,
            "y_val_raw": y_val,
            "y_test_raw": y_test,
            "df_augmented": df_augmented,
            "df_balanced": df_balanced,
            "x_train_dict": x_train_dict,
            "x_val_dict": x_val_dict,
            "x_test_dict": x_test_dict,
            "y_train_dict": y_train_dict,
            "y_val_dict": y_val_dict,
            "y_test_dict": y_test_dict,
            "x_scalers": x_scalers,
            "y_scalers": y_scalers
        }

    return all_outputs

In [ ]:
outputs = single_split_preprocessing(...)

In [ ]:
for task_name, out in outputs.items():

    X_train = out["X_train_raw"].to_numpy()
    Y_train = out["y_train_raw"].to_numpy()

    X_val = out["X_val_raw"].to_numpy()
    Y_val = out["y_val_raw"].to_numpy()

    X_test = out["X_test_raw"].to_numpy()
    Y_test = out["y_test_raw"].to_numpy()

In [ ]:
out = outputs["Multi_Task"]


X_train = out["X_train_raw"].to_numpy()
Y_train = out["y_train_raw"].to_numpy()

X_val = out["X_val_raw"].to_numpy()
Y_val = out["y_val_raw"].to_numpy()

X_test = out["X_test_raw"].to_numpy()
Y_test = out["y_test_raw"].to_numpy()

# versione senza for

In [ ]:
def single_split_preprocessing(
    features_dict,
    targets_dict,
    tasks_unified,
    output_path,
    scaler_path,
    n_splits=5,
    random_state=42
):
    task_name = "Multi_Task"
    #task_info = tasks_unified[task_name]

    printmd(f"# Task: {task_name}")
    printmd("---")

    X = features_dict
    y = targets_dict

    # =====================================
    # STESSO SPLIT DELLA CROSS_VALIDATE
    # ma uso solo il primo fold
    # =====================================
    print(f"Start CV Split (single fold only): {datetime.datetime.now()}")
    splits = get_folds(X, y, n_splits=n_splits, random_state=random_state)

    train_idx, test_idx = next(splits)

    # Outer split: train/test
    X_train_outer, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train_outer, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # =====================================
    # STESSO SPLIT TRAIN/VAL DELLA CROSS_VALIDATE
    # =====================================
    print(f"Start Train/Val Split: {datetime.datetime.now()}")
    inner_train_idx, inner_val_idx = stratified_group_shuffle_split(
        X_train_outer,
        y_train_outer.values[:, -1].astype(int)
    )

    X_train = X_train_outer.loc[inner_train_idx]
    y_train = y_train_outer.loc[inner_train_idx]
    X_val = X_train_outer.loc[inner_val_idx]
    y_val = y_train_outer.loc[inner_val_idx]

    # =====================================
    # OVERSAMPLING SOLO TRAIN
    # =====================================
    print(f"Start Oversampling: {datetime.datetime.now()}")
    df_train = pd.concat([X_train, y_train], axis=1)

    X_train, y_train, df_augmented, df_balanced = perform_custom_oversampling(
        df_train,
        tasks_unified["features"],
        tasks_unified["regression_targets"],
        tasks_unified["classification_targets"]
    )

    # =====================================
    # SALVA SPLIT RAW
    # =====================================
    save_xy_split(
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        output_path
    )

    # =====================================
    # PREPROCESSING
    # =====================================
    print(f"Start Preprocessing: {datetime.datetime.now()}")

    (
        x_train_dict, x_val_dict, x_test_dict, x_scalers,
        y_train_dict, y_val_dict, y_test_dict, y_scalers
    ) = preprocess_fold(
        X_train, X_val, X_test,
        y_train, y_val, y_test,
        tasks_unified,
        scaler_path
    )

    all_outputs = {
        "X_train_raw": X_train,
        "X_val_raw": X_val,
        "X_test_raw": X_test,
        "y_train_raw": y_train,
        "y_val_raw": y_val,
        "y_test_raw": y_test,
        "df_augmented": df_augmented,
        "df_balanced": df_balanced,
        "x_train_dict": x_train_dict,
        "x_val_dict": x_val_dict,
        "x_test_dict": x_test_dict,
        "y_train_dict": y_train_dict,
        "y_val_dict": y_val_dict,
        "y_test_dict": y_test_dict,
        "x_scalers": x_scalers,
        "y_scalers": y_scalers
    }

    return all_outputs

# missing
- get_folds
- stratified_group_shuffle_split
- save_xy_split
- training
- plot_training_histories_separate
- predict
- show_result
- cleanup_fold_memory